# mlflow_wrapper — example

A `Store` is scoped to one `use_case` and gives versioned parquet datasets, pyfunc models, and runs under a standardized name.

Config precedence: **arg > env var > default** (`MLFLOW_TRACKING_URI`, `MLFLOW_REGISTRY_URI`, default `file:./mlruns`).

In [ ]:
import pandas as pd
import mlflow
from mlflow_wrapper import Store

store = Store("forecasting")   # tracking_uri=... to override env/default
store.tracking_uri, store.registry_uri

## Datasets — submit by version, retrieve latest or a pinned version

In [ ]:
v1 = store.submit_dataset(pd.DataFrame({"x": [1, 2]}), "sales")
v2 = store.submit_dataset(pd.DataFrame({"x": [1, 2, 3]}), "sales")
print("versions:", store.list_dataset_versions("sales"))

store.get_dataset("sales")            # latest (v2)

In [ ]:
store.get_dataset("sales", version=1)  # pinned

## Runs — standardized name + use_case, params and metrics

In [ ]:
rid = store.submit_run("baseline", params={"lr": 0.1}, metrics={"rmse": 2.5})
run = store.get_run("baseline")
print(run.info.run_id, run.data.params, run.data.metrics)

store.list_runs()[["run_id", "tags.mlflow.runName", "tags.type"]]

## Models — auto-detected flavor, registry-versioned

`submit_model` picks the flavor from the object: **xgboost**, **sklearn**, and **sentence-transformers** log with their native flavor; anything else (a `mlflow.pyfunc.PythonModel`) goes through pyfunc. All load back through `get_model`.

In [ ]:
from sklearn.linear_model import LinearRegression

mv1 = store.submit_model(LinearRegression().fit([[0], [1], [2]], [0, 1, 2]), "ranker")
mv2 = store.submit_model(LinearRegression().fit([[0], [1], [2]], [0, 2, 4]), "ranker")
print("model versions:", mv1, mv2)

model = store.get_model("ranker")      # latest
model.predict(pd.DataFrame({"f": [1.5]}))

In [ ]:
# a model that builds on another one: record the base model + version for lineage
class Wrapper(mlflow.pyfunc.PythonModel):
    def predict(self, context, model_input, params=None):
        return model_input

store.submit_model(Wrapper(), "reranker", base_model="ranker", base_version=mv2)